In [13]:
import ollama

def ask_question_local_llm(prompt):
    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create

    # Run a prompt against a local model (e.g., llama2)
    response = ollama.chat(
        model='llama3',
        messages=[
            {"role": "system", "content": "You are a helpful AI assitant - Respond in one line"},
            {"role": "user", "content": prompt}
        ]
    )
    return response['message']['content']

In [14]:

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")


In [15]:

my_client = OpenAI(api_key=my_api_key)
# my_client

def ask_question_open_gpt_41(prompt):

    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create

    llm_response = my_client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
            {"role": "user", "content": prompt}
        ]
    )
    return llm_response.choices[0].message.content  


def ask_question_open_gpt_5(prompt):

    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create

    llm_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
            {"role": "user", "content": prompt}
        ]
    )
    return llm_response.choices[0].message.content  


In [16]:
prompt = "What is the capital of France?"

response_local_llm = ask_question_local_llm(prompt)
response_open_ai_41 = ask_question_open_gpt_41(prompt)
response_open_ai_5 = ask_question_open_gpt_5(prompt)

print(f"Response from local LLM: {response_local_llm}")
print(f"Response from OpenAI GPT-4.1: {response_open_ai_41}")
print(f"Response from OpenAI GPT-5: {response_open_ai_5}")

Response from local LLM: The capital of France is Paris.
Response from OpenAI GPT-4.1: The capital of France is Paris.
Response from OpenAI GPT-5: Paris.


In [17]:
import time
while True:
    # Ask user for a question
    user_prompt = input("Ask something: ")

    if (user_prompt.lower() != 'quit'):
        # Get and print the response
        
        print ("\n\n-------------------LOCAL LLM RESPONSE-------------------")
        start = time.time()
        response_local = ask_question_local_llm(user_prompt)
        end = time.time()  
        total_time_local = end - start 
        print(f"Time taken by Local LLM: {total_time_local} seconds")
        print("\nLocal LLM says:", response_local)     

        print ("-------------------OPEN AI RESPONSE - GPT-4.1 -------------------")
        start = time.time()
        response_open_ai_41 = ask_question_open_gpt_41(user_prompt)
        end = time.time()   
        total_time_open_ai_41 = end - start
        print(f"Time taken by OpenAI GPT-4.1: {total_time_open_ai_41} seconds")
        print("\nOpenAI GPT-4.1 says:", response_open_ai_41)

        print ("\n\n-------------------OPEN AI RESPONSE - GPT-5 -------------------")
        start = time.time()
        response_openai_5 = ask_question_open_gpt_5(user_prompt)
        end = time.time()
        total_time_openai_5 = end - start
        print(f"Time taken by OpenAI GPT-5: {total_time_openai_5} seconds")
        print("\nOpenAI GPT-5 says:", response_openai_5)

           

        # add delay of 3 seconds
        time.sleep(3)
    else:
        print("Exiting...")
        break    



-------------------LOCAL LLM RESPONSE-------------------
Time taken by Local LLM: 45.363736152648926 seconds

Local LLM says: Here is the analysis in the required format:

**SECTION 1 — Executive Summary**
The West Coast operations team reported a 23% decrease in inbound shipment processing across three warehouse locations due to issues with the new inventory synchronization service deployed on May 3rd.

**SECTION 2 — Root Cause Analysis**

* **Primary Cause:** Redis cache hit rate dropped from 91% to 63%, leading to increased PostgreSQL write throughput and API timeouts.
* **Secondary Causes:**
	+ Duplicate inventory records in Reno (12% of active SKUs).
	+ Delayed barcode scan propagation in Phoenix.
* **Evidence:**
	+ Reduced Redis cache hit rate.
	+ Increased PostgreSQL write throughput.
	+ API timeout spikes in Oakland.
* **Unknowns:** Possible race condition during rollback in Oakland, but logs are incomplete.

**SECTION 3 — Risk Assessment**

* **Severity Score:** 7 (scale of 

In [18]:
def compare_responses():
    comparison_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": '''You are an evaluator. Please evaluate the quality of the response 
             by different models on the basis of time taken and response generated. 
             Time should be 40% weightage and response quality should be 60% weightage in the evaluation.
             Put as a JSON response with keys as model names and values as score out of 10. 
             Also add a justification for the score - show Pros and Cons. 
             Show how you assigned the score e.g, why you added a score and why you decudted a score'''},
            {"role": "user", "content": "Time taken by Local LLM: " +  str(total_time_local)+ " seconds\nLocal LLM response: " + response_local 
            +"\n\nTime taken by OpenAI GPT-4.1: " + str(total_time_open_ai_41) + " seconds\nOpenAI GPT-4.1 response: " + response_open_ai_41 
            + "\n\nTime taken by OpenAI GPT-5: " + str(total_time_openai_5) + " seconds\nOpenAI GPT-5 response: " + response_openai_5 }
        ]
    )
    return comparison_response.choices[0].message.content  


In [19]:
comparison = compare_responses()
print("\n\nComparison of responses:\n", comparison)



Comparison of responses:
 {
  "OpenAI GPT-4.1": {
    "score": 9.10,
    "justification": {
      "pros": [
        "Strong structure and clarity across sections (Executive Summary, RCA, Risk, Actions, Questions, JSON).",
        "Accurate inclusion of key incident signals (time correlation, metrics like Redis hit rate, PostgreSQL writes, API timeouts).",
        "Structured JSON section present and coherent with the narrative.",
        "Concise, actionable recommendations and plausible mitigation steps."
      ],
      "cons": [
        "Some metric specifics could be more consistently sourced (e.g., 340% write throughput vs other variants in other models).",
        "Confidence/uncertainty discussion is present but could be more explicit about data gaps."
      ],
      "scoring_rationale": "Time-based scoring favored the fastest response (10.08s) yielding a strong time component (4.0). Quality was rated high (8.5/10) based on completeness and clarity, resulting in a total of 9.10